# 5. Streaming Responses

Switch from a blocking `messages.create` to `messages.stream` so tokens print incrementally as Claude generates them. Shows both the raw event loop (commented) and the higher-level `text_stream` helper.

In [1]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
from rich import print
import json

client = Anthropic()


In [2]:
def object_print(object): 
    return print(json.dumps(object.model_dump(), indent=2, default=str))

In [3]:
import sys


class ChatBot:
    """
    You are a helpful assistant
    """
    model = "claude-haiku-4-5"
    temperature = 0.7
    
    def __init__(self):
        self.messages = list()
        self.system_prompt = self.__doc__.strip()
    
    def add_user_message(self, text):
        user_message = {
            "role": "user", 
            "content": text
        }
        self.messages.append(user_message)
    
    def add_assistant_message(self, text):
        assistant_message = {
            "role": "assistant", 
            "content": text
        }
        self.messages.append(assistant_message)
    
    def run(self):
        if not self.messages:
            intro_text = "How can I help you?"
            print(intro_text)
            
        while True:
            user_input = input("> ")
            if user_input.lower() in ["exit", "stop", "bye", "end"]:
                break
            self.add_user_message(user_input)

            combined_text = ""
            # # Capture every stream object
            # stream = client.messages.create(
            #     model=self.model,
            #     max_tokens=250,
            #     system=self.system_prompt,
            #     temperature=self.temperature,
            #     messages=self.messages,
            #     stream=True
            # )            
            # for event in stream:
            #     object_print(event)

            # Handle only text in stream
            with client.messages.stream(
                model=self.model,
                max_tokens=250,
                messages=self.messages
            ) as stream:
                for text in stream.text_stream:
                    combined_text += text
                    # raw stdout, not rich.print — rich emits one display per call in Jupyter
                    sys.stdout.write(text)
                    sys.stdout.flush()

                final_text = stream.get_final_message().content[0].text
                assert final_text == combined_text
                self.add_assistant_message(final_text)
            sys.stdout.write("\n")


In [4]:
chatbot = ChatBot()
chatbot.run()

How can I help you?

>  Where in the world is Carmen Sandiego?


That's the classic question from the educational game and TV series! The answer is always different depending on which game, show, or book you're referring to.

In the games and shows, Carmen Sandiego is typically somewhere unexpected around the world—that's the whole point of the adventure. The fun is in figuring out where she is based on clues you gather.

Are you asking about:
- A specific game or show episode?
- Just looking for a fun reference?
- Something else about the Carmen Sandiego franchise?

I'd be happy to help if you can give me more context!


>  bye
